# Non-RAG Document Transforms with Fed Beige Book

This notebook demonstrates how to use the new **non-RAG document transforms** to generate forward-looking forecasting questions from the Federal Reserve's [Beige Book](https://www.federalreserve.gov/monetarypolicy/beige-book-default.htm) reports and resolve them using subsequent reports.

**What are non-RAG transforms?** Unlike the RAG-based `QdrantContextGenerator` and `QdrantRAGLabeler` (which search across multiple documents using embeddings), the non-RAG transforms pass the entire document into context for generating context and labels:

- **`FileSetDocumentContextGenerator`** — Finds the next (or previous) chronological document and appends its full text as context
- **`FileSetDocumentLabeler`** — Finds the next (or previous) chronological document and uses an LLM to extract a structured label from it

This is ideal for time-series document collections, where the entire document fits into context, and where each report naturally resolves questions raised by the previous one.

**Pipeline overview:**
1. Download 6 Beige Book PDFs from the Fed website
2. Create a FileSet and upload PDFs with date metadata
3. Generate seeds and forward-looking questions (`QuestionPipeline`)
4. Add context from the next document (`FileSetDocumentContextGenerator`)
5. Label using the next document (`FileSetDocumentLabeler`)
6. Inspect results

In [ ]:
%pip install -e lightningrod-ai python-dotenv requests pandas -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [7]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Download Beige Book PDFs

The Fed publishes Beige Book reports as PDFs at a predictable URL. We download 6 consecutive reports (Sep 2024 – Apr 2025) to keep the example quick and inexpensive while still having 5 valid temporal pairs for document resolution.

In [8]:
import requests
import tempfile
from pathlib import Path

BEIGE_BOOK_DATES = [
    "20240904",
    "20241023",
    "20241204",
    "20250115",
    "20250305",
    "20250423",
]

BASE_URL = "https://www.federalreserve.gov/monetarypolicy/files/BeigeBook_{date}.pdf"

pdf_dir = Path(tempfile.mkdtemp())

for date_str in BEIGE_BOOK_DATES:
    url = BASE_URL.format(date=date_str)
    filename = f"BeigeBook_{date_str}.pdf"
    out_path = pdf_dir / filename

    response = requests.get(url)
    response.raise_for_status()
    out_path.write_bytes(response.content)
    print(f"Downloaded: {filename} ({len(response.content) / 1024:.0f} KB)")

print(f"\nSaved {len(BEIGE_BOOK_DATES)} PDFs to {pdf_dir}")

Downloaded: BeigeBook_20240904.pdf (1006 KB)
Downloaded: BeigeBook_20241023.pdf (1019 KB)
Downloaded: BeigeBook_20241204.pdf (1030 KB)
Downloaded: BeigeBook_20250115.pdf (1013 KB)
Downloaded: BeigeBook_20250305.pdf (1091 KB)
Downloaded: BeigeBook_20250423.pdf (1096 KB)

Saved 6 PDFs to /var/folders/n5/1nmlnrf90y7dthhl9pk5w8h80000gn/T/tmpigshzec_


## Create FileSet and Upload PDFs

We create a FileSet for document-level transforms. Each document's `file_date` determines its position in the chronological sequence for temporal resolution.

The `upload_files()` utility handles parallel uploads and metadata manifest creation automatically.

In [9]:
fileset = lr.filesets.create(
    name="Beige Book",
    description="6 Federal Reserve Beige Book PDFs for non-RAG document processing transform demo",
)
print(f"Created FileSet: {fileset.id}")

Created FileSet: 0a4a18dc-7759-4440-9670-0e597d4507ac


In [10]:
from datetime import datetime, timezone

# Prepare file paths and metadata with file_date for temporal ordering
pdf_files = sorted(pdf_dir.glob("BeigeBook_*.pdf"))
metadata = {}
for pdf_path in pdf_files:
    date_str = pdf_path.stem.replace("BeigeBook_", "")
    file_date = datetime.strptime(date_str, "%Y%m%d").replace(tzinfo=timezone.utc)
    metadata[pdf_path.name] = {"file_date": file_date}

# Upload with utility (handles parallel uploads + manifest)
result = lr.filesets.upload_files(
    fileset.id,
    file_paths=[str(p) for p in pdf_files],
    metadata=metadata,
)

print(f"Uploaded {result.succeeded} files ({result.failed} failed)")
if result.errors:
    for error in result.errors:
        print(f"  Error: {error}")

Uploaded 6 files (0 failed)


## Build and execute pipeline

1. Generate seeds from uploaded fileset
2. Generate context directly from documents
3. Generate labels from the document immediately after the seed document

These questions ask about outcomes that can be verified by reading the *next* Beige Book report.

In [11]:
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    ForwardLookingQuestionGenerator,
    BinaryAnswerType,
    FileSetDocumentContextGenerator,
    TemporalConstraint,
    FileSetDocumentLabeler,
)

answer_type = BinaryAnswerType()

seed_generator = FileSetSeedGenerator(
    file_set_id=fileset.id,
    chunk_size=4000,
    chunk_overlap=200,
)

question_generator = ForwardLookingQuestionGenerator(
    questions_per_seed=5,
    answer_type=answer_type,
    instructions=(
        "Generate questions about whether specific economic outcomes will occur "
        "(decrease, increase, slow, accelerate, etc.). "
        "Ask about the outcome directly - e.g. 'Will loan nonperformance in Dallas decrease?' "
        "- NOT 'Will the next Beige Book report that...'. "
        "Do NOT use explicit dates, months, or years in the question or resolution criteria. "
        "Focus on district-specific topics (Dallas, St. Louis, Boston, etc.) and metrics that "
        "the Beige Book explicitly reports on. "
    ),
    examples=[
        "Will loan nonperformance in the Dallas district decrease?",
        "Will employment growth in the Philadelphia district slow?",
        "Will manufacturing activity in the St. Louis district improve?",
    ],
)

context_config = FileSetDocumentContextGenerator(
    file_set_id=fileset.id,
    temporal_constraint=TemporalConstraint.EQUAL,
    system_instruction=(
        "Extract the sections most relevant to economic forecasting from this "
        "Beige Book report. Focus on district-specific economic conditions, "
        "employment, prices, and outlook."
    ),
)

labeler_config = FileSetDocumentLabeler(
    file_set_id=fileset.id,
    temporal_constraint=TemporalConstraint.NEXT_DOCUMENT,
    confidence_threshold=0.7,
    answer_type=BinaryAnswerType(
        labeler_instruction=(
            "The provided document is always the correct resolution document. "
            "Resolve Yes or No only when the topic is explicitly addressed; "
            "if not reported, resolve as Undetermined. "
            "For increase/decrease questions: flat, stable, unchanged = No."
        ),
    ),
    system_instruction=(
        "You are labeling Federal Reserve forecasting questions. "
        "The provided document is the next chronological Beige Book report "
        "after the one that generated the question."
    ),
)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    context_generators=[context_config],
    labeler=labeler_config,
    
)

dataset_phase1 = lr.transforms.run(
    pipeline,
    max_questions=30,
    name="Beige Book Example",
)
print(f"Dataset: {dataset_phase1.id}")
print(f"Rows: {dataset_phase1.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           705f1716-8d6c-4e25-9dda-77292678cb92                                                       │
│                                                                                                                 │
│    Total cost: $0.62                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃ In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons   ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ FileSetSeedGenera… │ Complete             │  1 │  31 │        0 │      0 │ -                   │       4s │  │
│  │ ForwardLookingQue… │ Complete             │ 31 │ 155 │        0 │      0 │ -                   │       8s │  │
│  │ FileSetDocumentLa… │ Complete             │ 60 │  59 │        1 │      0 │ Undetermined label  │      24s │  │
│  │                    │                      │    │     │          │        │ (1)                 │          │  │
│  │ FileSetDocumentCo… │ Complete             │ 59 │  59 │        0 │      0 │ -                   │       6s │  │
│  └────────────────────┴──────────────────────┴────┴─────┴──────────┴────────┴─────────────────────┴──────────┘  │
│                                                                                                                 │
│    View full details:                                                                                           │
│  ]8;id=564984;https://dashboard.lightningrod.ai/?redirect=/datasets/757cdc30-0a50-4670-9514-317a3424258a\https://dashboard.lightningrod.ai/?redirect=/datasets/757cdc30-0a50-4670-9514-317a3424258a]8;;\                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dataset: 757cdc30-0a50-4670-9514-317a3424258a
Rows: 60


In [15]:
# Quick look at the generated questions
samples_p1 = dataset_phase1.download()
valid_samples = [s for s in samples_p1 if s.is_valid]
for i, s in enumerate(valid_samples[:5]):
    print(f"--- Sample {i+1} ---")
    print(f"Question: {s.question.question_text}")
    print(f"Date: {s.seed.seed_creation_date}")
    print(f"Label: {s.label.label if s.label else 'No label'} (confidence: {s.label.label_confidence if s.label else 'N/A'})")
    print()

--- Sample 1 ---
Question: Will loan demand in the Dallas district increase?
Date: 2024-09-04 00:00:00
Label: No (confidence: 1.0)

--- Sample 2 ---
Question: Will demand for professional and business services in the Cleveland district grow?
Date: 2024-09-04 00:00:00
Label: Yes (confidence: 0.9)

--- Sample 3 ---
Question: Will employment in the Philadelphia district nonmanufacturing sector increase?
Date: 2024-09-04 00:00:00
Label: No (confidence: 1.0)

--- Sample 4 ---
Question: Will residential construction activity in the Kansas City Federal Reserve District rise?
Date: 2024-09-04 00:00:00
Label: No (confidence: 1.0)

--- Sample 5 ---
Question: Will credit quality of borrowers in the Richmond district decrease?
Date: 2024-09-04 00:00:00
Label: No (confidence: 1.0)

